# Local multi-turn SQL lab

This notebook is the shareable lab for the post. It runs a small in-memory SQLite experiment that compares candidate fine-tuning targets for multi-turn SQL analysis.

The lab defaults to CPU. It can report CUDA, MPS, or XPU availability through PyTorch, but the SQLite experiment stays CPU-safe and does not require GPU compute.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "notebooks").exists():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").exists():
            repo_root = candidate
            break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
from notebooks.blog_support import data_engineering_gates
from notebooks.labs.local_multiturn_sql_lab_support import run_multiturn_lab


## 1. Research question

Can a small specialized model learn the behavior and semantic concepts needed to compete with hosted systems on multi-turn data analysis?

In [ ]:
report = run_multiturn_lab(device_preference="cpu")

print(f"Runtime used by the lab: {report['device'].label}")
print(f"Accelerator availability reported only: {report['detected_accelerator'].label}")
print("CUDA/MPS/XPU status:")
for status in report["accelerator_report"]:
    print(f"- {status['label']} ({status['usage']})")
print(f"Scenario hash: {report['scenario_contract']['shared_input_sha256']}")

assert report["device"].kind == "cpu"
assert report["runtime_policy"]["accelerator_usage"] == "reported_only"


## 2. Why single-turn SQL fails here

A single complete question can often be answered with one SQL query. A conversational analysis has to carry state across turns: the metric, filter, grain, value mapping, and recovery state can all change independently.

In [ ]:
import pandas as pd

pd.DataFrame(report["walkthrough_sections"])[
    ["section_id", "reader_question", "takeaway", "next_artifact"]
]


## 3. Candidate fine-tuning targets

In [ ]:
pd.DataFrame(report["method_matrix"])


## 4. Scores from the four-turn lab

In [ ]:
pd.DataFrame([
    {"system": system, **metrics}
    for system, metrics in report["systems"].items()
])


## 5. Execution trace

The trace separates value correctness from context carryover, value grounding, metric preservation, and recovery after an empty-result turn.

In [ ]:
trace_columns = [
    "turn_id",
    "question",
    "system",
    "value_match",
    "context_carryover",
    "value_grounded",
    "measure_preserved",
    "recovery_success",
    "failure_type",
    "intermediate_plan",
    "sql",
]

pd.DataFrame(report["rows"])[trace_columns]


## 6. Data engineering gates

These are the artifacts needed before the lab's method comparison can support a broader multi-turn SQL claim.

In [ ]:
data_engineering_gates()
